# CurAT-LSTM-FCN — Analysis: tables, statistics and figures
Reproduces every table, statistic and data-driven figure of the paper **from one file: `results_multiseed.xlsx`**
(columns `dataset, model, seed, clean, fgsm, bim, pgd, cw, mean_attack`; one row per model and seed, produced by `01_pipeline.ipynb` and merged over the 15 datasets).
No GPU, raw data or retraining is needed.

| Section | Reproduces |
|---|---|
| 3 | Table 1 (datasets) |
| 4 | Table 3 and Figures 4, 5, 6 (overall accuracy) |
| 5 | Table 4 and Figure 7 (per-dataset margins, accuracy drop) |
| 6 | Tables 5 and 6 |
| 7 | Figure 8 (series length vs. robustness) |
| 8 | Table 7 and Figure 9 (seed stability), Figure 3 (training curves) |
| 9 | Statistical tests (Wilcoxon + Holm, Friedman, effect sizes, Spearman, bootstrap) and Figure 10 |
| 10 | Appendix table (all 165 records) |

Seeded models (LSTM-FCN Vanilla, CurAT-LSTM-FCN) are averaged over seeds 0–3 first; the unit of analysis is the dataset (n = 15).
Figures 1 and 2 are drawings and are not generated here. Figures 3 and 9 need the training-history files of ECG5000 (see the configuration cell); they are skipped if those files are not found.
All outputs are written to `outputs/figures` (PNG, SVG, PDF) and `outputs/tables` (CSV, plus one Excel workbook).

## 1. Configuration

In [ ]:
import os, json, itertools, warnings
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
from scipy import stats
warnings.filterwarnings("ignore")

RESULTS_XLSX = "results_multiseed.xlsx"      # master results file (same folder as this notebook)
OUT_DIR      = "outputs"
FIG_DIR      = f"{OUT_DIR}/figures"
TAB_DIR      = f"{OUT_DIR}/tables"
# Training histories of ECG5000 (needed only for Figures 3 and 9). Set to None to skip them.
HIST_DIR     = "/content/drive/MyDrive/UCR_Datasets/ECG5000_checkpoints"

for d in (FIG_DIR, TAB_DIR): os.makedirs(d, exist_ok=True)
matplotlib.rcParams.update({"font.size": 12, "axes.titlesize": 14, "axes.labelsize": 12, "legend.fontsize": 10})

MODELS = ["ResNet", "FCN", "LSTM", "LSTM-FCN Vanilla", "CurAT-LSTM-FCN"]
V, C, L = "LSTM-FCN Vanilla", "CurAT-LSTM-FCN", "LSTM"
ATTACKS = ["fgsm", "bim", "pgd", "cw"]
ATK_LABEL = {"clean": "Clean", "fgsm": "FGSM", "bim": "BIM", "pgd": "PGD", "cw": "C&W"}
PALETTE = ["#4C72B0", "#DD8452", "#55A868", "#C44E52", "#8172B3"]    # one colour per model, in MODELS order

def save_fig(fig, name):
    for ext in ("png", "svg", "pdf"):
        fig.savefig(f"{FIG_DIR}/{name}.{ext}", dpi=300, bbox_inches="tight")
    print(f"saved {FIG_DIR}/{name}.[png|svg|pdf]")

TABLES = {}                                   # name -> DataFrame, written to Excel at the end
def keep(name, tab):
    TABLES[name] = tab
    tab.to_csv(f"{TAB_DIR}/{name}.csv", index=False)
    return tab

## 2. Load the results

In [ ]:
df = pd.read_excel(RESULTS_XLSX)
METRICS = ["clean", "fgsm", "bim", "pgd", "cw", "mean_attack"]

assert df.dataset.nunique() == 15, f"expected 15 datasets, found {df.dataset.nunique()}"
assert len(df) == 165, f"expected 165 rows (15 datasets x (3 + 2 models x 4 seeds)), found {len(df)}"
print("rows:", len(df), "| datasets:", df.dataset.nunique())
print(df.groupby("model")["seed"].apply(lambda x: sorted(x.dropna().unique().tolist())).to_string())

g  = df.groupby(["dataset", "model"])[METRICS].mean() * 100     # seed-averaged, in %
pa = g["mean_attack"].unstack()[MODELS]                         # dataset x model, mean-attack accuracy (%)
print("\nseed-averaged table:", g.shape, "| per-dataset mean-attack table:", pa.shape)

## 3. Table 1 — Datasets

In [ ]:
LENGTH = {"ECG5000": 140, "NonInvasiveFetalECGThorax1": 750, "EthanolLevel": 1751, "SemgHandMovementCh2": 1500,
          "FaceAll": 131, "FiftyWords": 270, "MixedShapesRegularTrain": 1024, "PhalangesOutlinesCorrect": 80,
          "SwedishLeaf": 128, "ShapesAll": 512, "FordA": 500, "FordB": 500, "MelbournePedestrian": 24,
          "TwoPatterns": 128, "ChlorineConcentration": 166}
table1 = pd.DataFrame([
    ("ECG5000", "Electrocardiography", 5, 140, 500, 4500),
    ("NonInvasiveFetalECGThorax1", "Electrocardiography", 42, 750, 1800, 1965),
    ("EthanolLevel", "Spectroscopy", 4, 1751, 504, 500),
    ("SemgHandMovementCh2", "Spectroscopy", 6, 1500, 450, 450),
    ("FaceAll", "Image-derived contour", 14, 131, 560, 1690),
    ("FiftyWords", "Image-derived contour", 50, 270, 450, 455),
    ("MixedShapesRegularTrain", "Image-derived contour", 5, 1024, 500, 2425),
    ("PhalangesOutlinesCorrect", "Image-derived contour", 2, 80, 1800, 858),
    ("SwedishLeaf", "Image-derived contour", 15, 128, 500, 625),
    ("ShapesAll", "Image-derived contour", 60, 512, 600, 600),
    ("FordA", "Industrial sensor", 2, 500, 3601, 1320),
    ("FordB", "Industrial sensor", 2, 500, 3636, 810),
    ("MelbournePedestrian", "Traffic / motion", 10, 24, 1194, 2439),
    ("TwoPatterns", "Simulated signal", 4, 128, 1000, 4000),
    ("ChlorineConcentration", "Simulated signal", 3, 166, 467, 3840),
], columns=["Dataset", "Domain", "Classes", "Series Length", "Train", "Test"])
assert all(LENGTH[d] == t for d, t in zip(table1.Dataset, table1["Series Length"]))
DS_ORDER = table1.Dataset.tolist()                 # dataset order used in the paper's heatmap
keep("Table1_datasets", table1)

## 4. Table 3 — Overall clean and adversarial accuracy (mean over 15 datasets, %)

In [ ]:
table3 = g.groupby("model").mean().loc[MODELS, METRICS].round(2)
table3.columns = ["Clean", "FGSM", "BIM", "PGD", "C&W", "Mean-Attack"]
table3 = table3.reset_index().rename(columns={"model": "Model"})
keep("Table3_overall", table3)

## 5. Figure 4 — Grouped bar chart: clean and adversarial accuracy of the five models

In [ ]:
avg = g.groupby("model").mean().loc[MODELS]
cols = ["clean"] + ATTACKS
x = np.arange(len(MODELS)); w = 0.16
fig, ax = plt.subplots(figsize=(12, 5.4))
for k, c in enumerate(cols):
    ax.bar(x + (k - 2) * w, avg[c].values, w, label=ATK_LABEL[c], color=PALETTE[k])
ax.set_xticks(x); ax.set_xticklabels(MODELS, rotation=15)
ax.set_ylabel("Accuracy (%)"); ax.set_title("Clean and Adversarial Accuracy Across Five Models")
ax.legend(loc="upper right", fontsize=9); ax.set_xlim(-0.55, len(MODELS) - 0.45)
plt.tight_layout(); save_fig(fig, "Fig4_grouped_bar"); plt.show()

## 6. Figure 5 — Radar chart: clean and adversarial accuracy of the five models

In [ ]:
labels = [ATK_LABEL[c] for c in cols]
theta = np.linspace(0, 2 * np.pi, len(cols), endpoint=False)
fig = plt.figure(figsize=(9, 8.3)); ax = fig.add_subplot(111, polar=True)
for k, m in enumerate(MODELS):
    v = (avg.loc[m, cols].values / 100).tolist(); v += v[:1]; th = np.append(theta, theta[0])
    ax.plot(th, v, color=PALETTE[k], lw=2.5, label=m)
    ax.fill(th, v, color=PALETTE[k], alpha=0.07)
ax.set_xticks(theta); ax.set_xticklabels(labels, fontsize=16)
ax.set_ylim(0, 1); ax.set_yticks([0.2, 0.4, 0.6, 0.8]); ax.tick_params(axis="y", labelsize=14)
ax.set_title("Clean and Adversarial Accuracy Across Five Models", pad=40)
ax.legend(loc="upper left", bbox_to_anchor=(1.05, 1.0))
plt.tight_layout(); save_fig(fig, "Fig5_radar"); plt.show()

## 7. Figure 6 — Boxplot: distribution of mean-attack accuracy across the 15 datasets

In [ ]:
BOX_COL = ["#546E7A", "#5DB5AA", "#EDD07F", "#F5B785", "#E8917A"]
fig, ax = plt.subplots(figsize=(12, 5.2))
bp = ax.boxplot([pa[m].values for m in MODELS], patch_artist=True, widths=0.5, whis=1.5,
                medianprops=dict(color="black", lw=2.2), whiskerprops=dict(lw=1.6), capprops=dict(lw=1.6),
                boxprops=dict(lw=1.3), flierprops=dict(marker="o", markerfacecolor="white", markeredgecolor="black", markersize=6))
for patch, c in zip(bp["boxes"], BOX_COL): patch.set_facecolor(c); patch.set_alpha(0.9)
ax.set_xticklabels(MODELS, rotation=15); ax.set_ylabel("Mean-Attack Accuracy (%)")
ax.set_title("Distribution of Mean-Attack Accuracy Across 15 Datasets")
ax.set_facecolor("#FAFAFA"); ax.grid(axis="y", alpha=0.35)
plt.tight_layout(); save_fig(fig, "Fig6_boxplot"); plt.show()
print(pa.describe(percentiles=[.25, .5, .75]).T[["min", "25%", "50%", "75%", "max"]].round(2))

## 8. Table 4 — CurAT-LSTM-FCN vs LSTM-FCN Vanilla, mean-attack accuracy by dataset (%)

In [ ]:
table4 = pd.DataFrame({"Dataset": pa.index, "LSTM-FCN Vanilla": pa[V].values, "CurAT-LSTM-FCN": pa[C].values})
table4["Margin"] = table4["CurAT-LSTM-FCN"] - table4["LSTM-FCN Vanilla"]
print(f"CurAT wins on {(table4.Margin > 0).sum()}/15 datasets | mean margin {table4.Margin.mean():+.2f} | median {table4.Margin.median():+.2f}")
table4 = table4.sort_values("Margin", ascending=False).round(2).reset_index(drop=True)
keep("Table4_curat_vs_vanilla", table4)
table4

## 9. Figure 7 — Heatmap: accuracy drop (clean − attack) for LSTM-FCN Vanilla and CurAT-LSTM-FCN

In [ ]:
cols7, mat = [], []
drop = {m: {a: g.xs(m, level="model")["clean"] - g.xs(m, level="model")[a] for a in ATTACKS} for m in (V, C)}
for a in ATTACKS:
    cols7 += [f"{ATK_LABEL[a]}-Vanilla", f"{ATK_LABEL[a]}-CurAT"]
    mat += [drop[V][a].loc[DS_ORDER].values, drop[C][a].loc[DS_ORDER].values]
M = np.array(mat).T                                   # 15 datasets x 8 columns
fig, ax = plt.subplots(figsize=(12, 8.2))
im = ax.imshow(M, cmap="RdYlGn_r", aspect="auto")
ax.set_xticks(range(len(cols7))); ax.set_xticklabels(cols7, rotation=45, ha="right")
ax.set_yticks(range(len(DS_ORDER))); ax.set_yticklabels(DS_ORDER)
for i in range(M.shape[0]):
    for j in range(M.shape[1]): ax.text(j, i, f"{M[i, j]:.0f}", ha="center", va="center", fontsize=9)
cb = fig.colorbar(im, ax=ax); cb.set_label("Accuracy Drop (%)")
ax.set_title("Accuracy Drop (Clean − Attack) for LSTM-FCN Vanilla vs CurAT-LSTM-FCN")
plt.tight_layout(); save_fig(fig, "Fig7_accuracy_drop_heatmap"); plt.show()
# numbers cited in the Figure 7 paragraph
for a in ATTACKS:
    n = int((drop[C][a] < drop[V][a]).sum())
    print(f"{ATK_LABEL[a]:5}: CurAT has the smaller drop on {n:2}/15 datasets | mean drop Vanilla {drop[V][a].mean():5.1f}, CurAT {drop[C][a].mean():5.1f}")

## 10. Table 5 — FGSM vs C&W accuracy of CurAT-LSTM-FCN by dataset (%), sorted by |margin|

In [ ]:
cur = g.xs(C, level="model")
table5 = pd.DataFrame({"Dataset": cur.index, "FGSM": cur["fgsm"].values, "C&W": cur["cw"].values})
table5["Margin (C&W − FGSM)"] = table5["C&W"] - table5["FGSM"]
table5 = table5.reindex(table5["Margin (C&W − FGSM)"].abs().sort_values(ascending=False).index).round(2).reset_index(drop=True)
keep("Table5_fgsm_vs_cw_curat", table5)

## 11. Table 6 — CurAT-LSTM-FCN vs LSTM, mean-attack accuracy margin by dataset

In [ ]:
table6 = pd.DataFrame({"Dataset": pa.index, "Margin (CurAT-LSTM-FCN − LSTM)": (pa[C] - pa[L]).values})
table6["Winner"] = np.where(table6.iloc[:, 1] > 0, "CurAT-LSTM-FCN", "LSTM")
table6 = table6.sort_values("Margin (CurAT-LSTM-FCN − LSTM)", ascending=False).round(2).reset_index(drop=True)
keep("Table6_curat_vs_lstm", table6)
print((table6.Winner == "CurAT-LSTM-FCN").sum(), "/ 15 wins for CurAT-LSTM-FCN")
table6

## 12. Figure 8 — Series length vs. mean-attack accuracy (Spearman rank correlation), one panel per model

In [ ]:
try:
    from adjustText import adjust_text         # keeps point labels from overlapping
except ImportError:
    import subprocess, sys
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "adjustText"])
    try:
        from adjustText import adjust_text
    except Exception:
        adjust_text = None                     # labels may overlap, results are unaffected
T = pd.Series(LENGTH)[pa.index]
for m in MODELS:
    y = pa[m]; rho, p = stats.spearmanr(T, y)
    fig, ax = plt.subplots(figsize=(8, 5.6))
    ax.scatter(T, y, s=140, color="#1f77b4", edgecolor="black", zorder=3)
    k, b = np.polyfit(T, y, 1); xs = np.linspace(T.min(), T.max(), 100)
    ax.plot(xs, k * xs + b, "--", color="#1f77b4", alpha=0.6, zorder=2)
    texts = [ax.text(T[d], y[d], d, fontsize=9) for d in pa.index]
    if adjust_text: adjust_text(texts, ax=ax, arrowprops=dict(arrowstyle="-", color="gray", lw=0.6))
    ax.set_xlim(-200, 2000); ax.set_ylim(-10, 97)
    ax.set_xlabel("Series Length (T)"); ax.set_ylabel("Mean-Attack Accuracy (%)")
    ax.set_title(f"{m}   (ρ={rho:.3f}, p={p:.3f})", fontweight="bold")
    plt.tight_layout(); save_fig(fig, f"Fig8_length_vs_robustness_{m.replace(' ', '_')}"); plt.show()

## 13. Table 7 — Seed stability: mean ± SD of mean-attack accuracy over seeds 0–3 (%)

In [ ]:
seeded = df[df.model.isin([V, C])].groupby(["dataset", "model"])["mean_attack"].agg(["mean", "std"]) * 100   # sample SD (ddof = 1)
mu, sd = seeded["mean"].unstack(), seeded["std"].unstack()
table7 = pd.DataFrame({
    "Dataset": mu.index,
    "LSTM-FCN Vanilla (mean ± SD)": [f"{a:.2f} ± {b:.2f}" for a, b in zip(mu[V], sd[V])],
    "CurAT-LSTM-FCN (mean ± SD)":   [f"{a:.2f} ± {b:.2f}" for a, b in zip(mu[C], sd[C])],
    "Margin": [f"{a - b:+.2f}" for a, b in zip(mu[C], mu[V])],
    "_sd": sd[C].values}).sort_values("_sd", ascending=False).drop(columns="_sd").reset_index(drop=True)
keep("Table7_seed_stability", table7)
table7

## 14. Figure 3 and Figure 9 — Training curves (ECG5000; need the training-history files)

In [ ]:
def read_hist(names):
    for n in names:
        p = f"{HIST_DIR}/{n}"
        if HIST_DIR and os.path.exists(p):
            with open(p) as f: return json.load(f)
    return None

van = {s: read_hist([f"ECG5000_vanilla_seed{s}_history.json", f"vanilla_seed{s}_history.json"]) for s in range(4)}
cur_h = {s: read_hist([f"ECG5000_robust_fixed_seed{s}_history.json"]) for s in range(4)}
HAVE_HIST = HIST_DIR is not None and all(v is not None for v in list(van.values()) + list(cur_h.values()))
print("training histories found:", HAVE_HIST)
if not HAVE_HIST:
    print("Figures 3 and 9 skipped. Set HIST_DIR in the configuration cell to the ECG5000 checkpoint folder.")

In [ ]:
# ---- Figure 3: seed 0, Vanilla vs CurAT ----
if HAVE_HIST:
    C_V, C_C = "#003F5C", "#FF6361"
    for key, ylab, title, name in [("val_accuracy", "Validation accuracy", "(a) Validation accuracy", "Fig3a_validation_accuracy"),
                                   ("loss", "Training loss", "(b) Training loss", "Fig3b_training_loss")]:
        fig, ax = plt.subplots(figsize=(7.2, 3.6))
        ax.plot(van[0][key], color=C_V, lw=1.4, label="LSTM-FCN Vanilla")
        ax.plot(cur_h[0][key], color=C_C, lw=1.4, label="CurAT-LSTM-FCN")
        ax.set_xlabel("Epoch"); ax.set_ylabel(ylab); ax.set_title(title, fontsize=13); ax.grid(alpha=0.25)
        ax.legend(loc="lower right" if key == "val_accuracy" else "upper right")
        plt.tight_layout(); save_fig(fig, name); plt.show()

In [ ]:
# ---- Figure 9: all four seeds ----
if HAVE_HIST:
    SEED_COL = ["#8A2BE2", "#FF8C00", "#12B886", "#E03546"]
    panels = [("(a) Vanilla: validation accuracy", van, "val_accuracy", "Validation accuracy", "Fig9a"),
              ("(b) CurAT: validation accuracy", cur_h, "val_accuracy", "Validation accuracy", "Fig9b"),
              ("(c) Vanilla: training loss", van, "loss", "Training loss", "Fig9c"),
              ("(d) CurAT: training loss", cur_h, "loss", "Training loss", "Fig9d")]
    for title, H, key, ylab, name in panels:
        fig, ax = plt.subplots(figsize=(7.2, 3.4))
        for s in range(4): ax.plot(H[s][key], color=SEED_COL[s], lw=1.2, label=f"Seed {s}")
        ax.set_xlabel("Epoch"); ax.set_ylabel(ylab); ax.set_title(title, fontsize=13); ax.grid(alpha=0.25)
        if key == "val_accuracy": ax.set_ylim(0, 1)
        ax.legend(ncol=4, loc="lower right" if key == "val_accuracy" else "upper right")
        plt.tight_layout(); save_fig(fig, f"{name}_{key}"); plt.show()

## 15. Statistical tests
The cells below recompute every statistic quoted in the paper from the results file only.
Wilcoxon p-values are computed with SciPy and by exact enumeration of all 2ⁿ sign patterns. R equivalents:
`wilcox.test(x, y, paired=TRUE, exact=TRUE)`, `friedman.test`, `p.adjust(p, method="holm")`, `cor.test(T, y, method="spearman")`.

### 15.1 Sanity check, Table 3 and the exact Wilcoxon routine

In [ ]:
import itertools
from scipy import stats

PATH = RESULTS_XLSX
df = pd.read_excel(PATH)
MODELS = ['ResNet', 'FCN', 'LSTM', 'LSTM-FCN Vanilla', 'CurAT-LSTM-FCN']
V, C, L = MODELS[3], MODELS[4], MODELS[2]
LENGTH = {'ChlorineConcentration':166,'ECG5000':140,'EthanolLevel':1751,'FaceAll':131,'FiftyWords':270,'FordA':500,'FordB':500,
          'MelbournePedestrian':24,'MixedShapesRegularTrain':1024,'NonInvasiveFetalECGThorax1':750,'PhalangesOutlinesCorrect':80,
          'SemgHandMovementCh2':1500,'ShapesAll':512,'SwedishLeaf':128,'TwoPatterns':128}

# --- sanity: mean_attack must equal the mean of the four attacks in every row
bad = df[(df[['fgsm','bim','pgd','cw']].mean(axis=1) - df['mean_attack']).abs() > 0.0005]
print(f"rows={len(df)} | rows where mean_attack != mean of 4 attacks: {len(bad)}")
if len(bad): print(bad[['dataset','model','seed','fgsm','bim','pgd','cw','mean_attack']].to_string())

# --- seed-average the two seeded models first; unit of analysis = dataset (n = 15)
g = df.groupby(['dataset', 'model'])[['clean','fgsm','bim','pgd','cw','mean_attack']].mean() * 100
pa = g['mean_attack'].unstack()[MODELS]
print("\nTABLE 3 (average over 15 datasets, %):"); print(g.groupby('model').mean().loc[MODELS].round(2).to_string())

def exact_wilcoxon_p(x, y):
    d = np.asarray(x) - np.asarray(y); d = d[d != 0]; n = len(d); a = np.abs(d)
    order = a.argsort(kind='mergesort'); s = a[order]; r = np.empty(n); i = 0
    while i < n:
        j = i
        while j + 1 < n and s[j + 1] == s[i]: j += 1
        r[i:j + 1] = (i + j) / 2 + 1; i = j + 1
    ranks = np.empty(n); ranks[order] = r
    w = ranks[d > 0].sum(); rr = np.round(ranks * 2).astype(int); T = rr.sum()
    dp = np.zeros(T + 1); dp[0] = 1
    for v in rr: new = dp.copy(); new[v:] += dp[:T + 1 - v]; dp = new
    dp /= dp.sum(); k = np.arange(T + 1)
    return float(dp[np.abs(k - T / 2) >= abs(round(w * 2) - T / 2) - 1e-9].sum())

### 15.2 Pairwise Wilcoxon (Holm), Friedman, effect sizes, Spearman

In [ ]:
print("\nPAIRWISE WILCOXON (paired, two-sided) on mean-attack accuracy, Holm over 10 pairs")
pairs = list(itertools.combinations(MODELS, 2)); raw = [stats.wilcoxon(pa[a], pa[b]).pvalue for a, b in pairs]
order = np.argsort(raw); holm = np.zeros(10); run = 0
for rank, i in enumerate(order): run = max(run, raw[i] * (10 - rank)); holm[i] = min(run, 1)
for (a, b), p, h in zip(pairs, raw, holm):
    print(f"  {a:17} vs {b:17} scipy p={p:.4f}  exact p={exact_wilcoxon_p(pa[a], pa[b]):.4f}  Holm p={h:.4f}")

fr = stats.friedmanchisquare(*[pa[m] for m in MODELS])
print(f"\nFRIEDMAN: chi2={fr.statistic:.3f}, p={fr.pvalue:.2e}; average ranks (1 = best):", pa.rank(axis=1, ascending=False).mean().round(2).to_dict())

cliff = lambda x, y: (np.sum(np.asarray(x)[:, None] > np.asarray(y)[None, :]) - np.sum(np.asarray(x)[:, None] < np.asarray(y)[None, :])) / (len(x) * len(y))
print(f"\nCLIFF'S DELTA: CurAT vs Vanilla={cliff(pa[C], pa[V]):.3f}, CurAT vs LSTM={cliff(pa[C], pa[L]):.3f}")
def _rb(x, y):   # matched-pairs rank-biserial correlation of the Wilcoxon test = (W+ - W-) / (W+ + W-)
    d = (x - y).values; d = d[d != 0]; r = stats.rankdata(np.abs(d)); return (r[d > 0].sum() - r[d < 0].sum()) / r.sum()
print(f"MATCHED-PAIRS RANK-BISERIAL r: CurAT vs Vanilla={_rb(pa[C], pa[V]):.3f} ({int((pa[C] > pa[V]).sum())}/15 positive), CurAT vs LSTM={_rb(pa[C], pa[L]):.3f} ({int((pa[C] > pa[L]).sum())}/15 positive)")
print("CV across datasets (%):", {m: float(round(pa[m].std(ddof=1) / pa[m].mean() * 100, 2)) for m in MODELS})

t = pd.Series(LENGTH)[pa.index]
print("SPEARMAN (series length vs mean-attack):", {m: (float(round(stats.spearmanr(t, pa[m])[0], 3)), float(round(stats.spearmanr(t, pa[m])[1], 3))) for m in MODELS})

### 15.3 Per-attack paired Wilcoxon: CurAT-LSTM-FCN vs each comparator

In [ ]:
print("\nPER-ATTACK PAIRED WILCOXON (two-sided, uncorrected), CurAT-LSTM-FCN vs each comparator  [Sections 5.3 and 5.7]")
for comp in (V, L):
    print(f"  vs {comp}:")
    for atk in ['fgsm', 'bim', 'pgd', 'cw']:
        q = g[atk].unstack(); p = stats.wilcoxon(q[C], q[comp]).pvalue
        print(f"    {atk:5} CurAT higher on {int((q[C] > q[comp]).sum()):2}/15 | mean diff {(q[C] - q[comp]).mean():+7.2f} | "
              f"p={p:.4g}  exact p={exact_wilcoxon_p(q[C], q[comp]):.4g}")
cw_ex = g['cw'].unstack(); dd = (cw_ex[C] - cw_ex[V])
print(f"  C&W: CurAT minus Vanilla excluding TwoPatterns: mean {dd.drop('TwoPatterns').mean():+.2f}; TwoPatterns alone {dd['TwoPatterns']:+.1f}")

### 15.4 Numbers quoted in Sections 5.1–5.7 (boxplot statistics, margins, seed variability, consistency, CurAT vs LSTM, C&W vs FGSM)

In [ ]:
print("\nSECTION 5.1  (boxplot statistics of per-dataset mean-attack accuracy, %)")
bx = pa.describe(percentiles=[.25, .5, .75]).T[['min', '25%', '50%', '75%', 'max']]; bx['IQR'] = bx['75%'] - bx['25%']
print(bx.round(1).to_string())
av = g.groupby('model').mean().loc[MODELS]
print("lowest attack per model:", av[['fgsm', 'bim', 'pgd', 'cw']].idxmin(axis=1).to_dict())
print("spread across the 4 attacks (max - min, points):", (av[['fgsm', 'bim', 'pgd', 'cw']].max(axis=1) - av[['fgsm', 'bim', 'pgd', 'cw']].min(axis=1)).round(2).to_dict())
print(f"clean-accuracy cost of CurAT vs Vanilla: {av.loc[V, 'clean'] - av.loc[C, 'clean']:.4f} points (unrounded)")
print(f"mean-attack gain of CurAT vs Vanilla:    {av.loc[C, 'mean_attack'] - av.loc[V, 'mean_attack']:.4f} points (unrounded)")

print("\nSECTION 5.2  (CurAT minus Vanilla, mean-attack accuracy per dataset)")
mg = (pa[C] - pa[V]).sort_values(ascending=False); wn = mg[mg > 0]
print(f"  wins {len(wn)}/15 | mean over all 15 = {mg.mean():.2f} | median = {mg.median():.2f} | mean among winners = {wn.mean():.2f}")
print("  losses:", mg[mg < 0].round(2).to_dict(), "| margins under 5 points:", mg[(mg > 0) & (mg < 5)].round(2).to_dict())

print("\nSECTION 5.6  (seed-to-seed variability of mean-attack accuracy)")
s = df[df.model.isin([V, C])].groupby(['dataset', 'model'])['mean_attack'].std(ddof=1).unstack() * 100
print(f"  mean SD: Vanilla {s[V].mean():.2f}, CurAT {s[C].mean():.2f} | median: {s[V].median():.2f}, {s[C].median():.2f} | CurAT lower on {int((s[C] < s[V]).sum())}/15")
print(f"  paired Wilcoxon on the per-dataset SDs: p = {stats.wilcoxon(s[C], s[V]).pvalue:.3f}")
_r = stats.spearmanr(s[V], s[C]); print(f"  Spearman between the two models' per-dataset SDs: rho = {_r[0]:.3f}, p = {_r[1]:.3f}")
print("  SD > 5 points:", {m: s.index[s[m] > 5].tolist() for m in (V, C)})
_cl = df[(df.dataset == 'ECG5000') & df.model.isin([V, C])].groupby('model')['clean'].agg(['mean', 'std']) * 100
print("  ECG5000 clean accuracy (mean, SD):", _cl.round(2).to_dict('index'))

print("\nSECTION 5.5  (consistency across datasets)")
print("  SD of per-dataset mean-attack (points):", {m: float(round(pa[m].std(ddof=1), 2)) for m in MODELS})
rng = np.random.RandomState(0); idx = rng.randint(0, len(pa), size=(10000, len(pa)))   # fixed seed: same resamples every run
cvb = {m: pa[m].values[idx].std(axis=1, ddof=1) / pa[m].values[idx].mean(axis=1) * 100 for m in MODELS}
for other in (L, 'ResNet', 'FCN', V):
    dlt = cvb[C] - cvb[other]
    print(f"  paired bootstrap CV(CurAT) - CV({other}): {dlt.mean():6.1f}  95% CI [{np.percentile(dlt, 2.5):6.1f}, {np.percentile(dlt, 97.5):6.1f}]")
tc = stats.t.ppf(0.975, 13); print(f"  smallest |rho| reaching p<0.05 with n=15 (Spearman, t-approximation): {tc / np.sqrt(13 + tc**2):.3f}")

print("\nSECTION 5.4  (CurAT vs pure LSTM)")
print(f"  aggregate gap = {pa[C].mean() - pa[L].mean():.4f} | wins {int((pa[C] > pa[L]).sum())}/15 | losses:", (pa[C] - pa[L])[pa[C] < pa[L]].round(2).to_dict())
x = pa.drop('EthanolLevel'); print(f"  excluding EthanolLevel: CurAT wins {int((x[C] > x[L]).sum())}/14, Wilcoxon p = {stats.wilcoxon(x[C], x[L]).pvalue:.4f}")
cl = g['clean'].unstack()
for ds in ['TwoPatterns', 'ECG5000', 'FiftyWords']: print(f"  {ds:12} LSTM clean {cl.loc[ds, L]:.2f} mean-attack {pa.loc[ds, L]:.2f} | CurAT clean {cl.loc[ds, C]:.2f} mean-attack {pa.loc[ds, C]:.2f}")

print("\nSECTION 5.3  (C&W minus FGSM accuracy per model; negative = C&W lower)")
for m in MODELS:
    d = g.xs(m, level='model')['cw'] - g.xs(m, level='model')['fgsm']
    print(f"  {m:17} C&W below FGSM on {int((d < 0).sum()):2}/15 datasets | mean {d.mean():7.2f} | min {d.min():7.2f} | max {d.max():7.2f}")
dcur = g.xs(C, level='model')['cw'] - g.xs(C, level='model')['fgsm']
print("  CurAT extremes:", dcur.idxmax(), round(dcur.max(), 2), "|", dcur.idxmin(), round(dcur.min(), 2))

s = df[df.model.isin([V, C])].groupby(['dataset', 'model'])['mean_attack'].std(ddof=1).unstack() * 100
print(f"\nSEED SD of mean-attack (pp): mean Vanilla={s[V].mean():.2f}, CurAT={s[C].mean():.2f}; CurAT lower on {int((s[C] < s[V]).sum())}/15 datasets")
print(f"CurAT beats Vanilla (mean-attack) on {int((pa[C] > pa[V]).sum())}/15 datasets; CurAT beats LSTM on {int((pa[C] > pa[L]).sum())}/15")

## 16. Figure 10 — Friedman average ranks of mean-attack accuracy

In [ ]:
fr = stats.friedmanchisquare(*[pa[m] for m in MODELS])
ranks = pa.rank(axis=1, ascending=False).mean().sort_values()          # 1 = best
def sci(p):
    e = int(np.floor(np.log10(p))); mant = p / 10 ** e
    sup = str(e).replace("-", "⁻").translate(str.maketrans("0123456789", "⁰¹²³⁴⁵⁶⁷⁸⁹"))
    return f"{mant:.1f}×10{sup}"
fig, ax = plt.subplots(figsize=(10, 5.8))
yy = np.arange(len(ranks))[::-1]
for yi, (m, r) in zip(yy, ranks.items()):
    col = "#DD8452" if m == C else "#4C72B0"
    ax.hlines(yi, 0, r, color=col, lw=4.5, alpha=0.7)
    ax.plot(r, yi, "o", color=col, markersize=13, markeredgecolor="white", markeredgewidth=1.5)
    ax.text(r + 0.1, yi, f"{r:.2f}", va="center", fontweight="bold", fontsize=15)
ax.set_yticks(yy); ax.set_yticklabels(ranks.index, fontsize=14)
ax.set_xlim(0, 4.7); ax.set_xlabel("Average Rank (lower = better)", fontsize=14)
ax.set_title(f"Friedman Average Rank of Mean-Attack Accuracy\n(χ² = {fr.statistic:.2f}, p = {sci(fr.pvalue)})", fontsize=15)
ax.grid(axis="x", alpha=0.4); ax.set_axisbelow(True)
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout(); save_fig(fig, "Fig10_friedman_ranks"); plt.show()
print(ranks.round(2).to_dict())

## 17. Appendix table — all 165 records, and export of all tables to one Excel workbook

In [ ]:
appendix = df.copy()
appendix["seed"] = appendix["seed"].astype("Int64")
for c in ["clean", "fgsm", "bim", "pgd", "cw", "mean_attack"]: appendix[c] = (appendix[c] * 100).round(2)
appendix = appendix.rename(columns={"dataset": "Dataset", "model": "Model", "seed": "Seed", "clean": "Clean", "fgsm": "FGSM",
                                    "bim": "BIM", "pgd": "PGD", "cw": "C&W", "mean_attack": "Mean-Attack"})
appendix = appendix[["Dataset", "Model", "Seed", "Clean", "FGSM", "BIM", "PGD", "C&W", "Mean-Attack"]]
keep("TableA1_all_records", appendix)

with pd.ExcelWriter(f"{TAB_DIR}/all_tables.xlsx") as xw:
    for name, tab in TABLES.items(): tab.to_excel(xw, sheet_name=name[:31], index=False)
print("tables written to", TAB_DIR, "->", sorted(TABLES))